In [ ]:
class RecurrentGPT(torch.nn.Module):
    def __init__(
        self,
        config: RecurrentConfig,
        objective,
        gradient_checkpointing=False,
    ) -> None:
        super().__init__()
        assert config.padded_vocab_size is not None
        self.config = config

        # Transformer layers
        # Defines the beginning embedding of tokens
        prelude = torch.nn.ModuleList(config.Block(config, layer_id=i) for i in range(config.n_layers_in_prelude))

        if config.injection_type == "linear":
            adapter = config.Linear(
                config.n_embd * 2,
                config.n_embd,
                bias=config.bias,
                init_method=config.init.fn("in_proj", config.n_layers_in_prelude),
            )
        elif config.injection_type == "ffn":
            adapter = config.MLP(config, layer_id=0, in_features=config.n_embd * 2)
        else:
            adapter = torch.nn.Identity()

        # this block will be recalled
        core_block = torch.nn.ModuleList(
            config.Block(config, layer_id=i + config.n_layers_in_prelude)
            for i in range(config.n_layers_in_recurrent_block)
        )
        o = config.n_layers_in_prelude + config.n_layers_in_recurrent_block * config.mean_recurrence
        coda = torch.nn.ModuleList(config.Block(config, layer_id=i + o) for i in range(config.n_layers_in_coda))

        hidden_state_dim = config.n_embd if config.Block is not RevTransformerPreNormBlock else config.n_embd * 2
        self.transformer = torch.nn.ModuleDict(
            dict(
                wte=torch.nn.Embedding(config.padded_vocab_size, hidden_state_dim),
                prelude=prelude,
                adapter=adapter,
                core_block=core_block,
                coda=coda,
                ln_f=config.Norm(hidden_state_dim, eps=config.norm_eps),
            )
        )
        self.emb_scale = config.init.embedding_scale
        # Head
        if config.use_fused_head == "cce":
            self.lm_head = config.Linear(
                hidden_state_dim, config.padded_vocab_size, bias=False, init_method=config.init.fn("head")
            )
        elif config.use_fused_head == "hhe":
            from recpre.utils import LinearCrossEntropyLoss as LCE

            self.lm_head = LCE(
                hidden_state_dim,
                config.padded_vocab_size,
                ignore_index=objective["ignore_index"],
                init_method=config.init.fn("head"),
            )
        elif self.config.use_fused_head == "full-triton":
            self.lm_head = LinearCrossEntropyLoss(
                hidden_state_dim,
                config.padded_vocab_size,
                ignore_index=objective["ignore_index"],
                z_regularization=objective["z_regularization"],
                logit_scale=config.init.logit_scale,
                init_method=config.init.fn("head"),
                transposed_weight=not self.config.tie_embeddings,
            )
        else:
            self.lm_head = config.Linear(
                hidden_state_dim, config.padded_vocab_size, bias=False, init_method=config.init.fn("head")
            )
        if self.config.tie_embeddings:
            self.lm_head.weight = self.transformer.wte.weight
        self.objective = objective

        # rarely used features:
        if config.embed_step:
            self.step_embedding = TimestepEmbedder(config.n_embd)

        # Misc attributes
        self.max_seq_length = self.config.block_size
        self.gradient_checkpointing = gradient_checkpointing
        self.register_buffer("freqs_cis", self._precompute_freqs_cis(), persistent=True)

        # Externally set:
        self.step = 0
        self.monitoring = False
        self.latest_metrics = {}
        # Remaining inits:
        self.reset_parameters()

    def _precompute_freqs_cis(self):
        # Trigger resetting the rope-cache
        dim = self.config.intermediate_size if self.transformer.core_block[0].expanded else self.config.n_embd
        if self.config.randomize_positions_from is not None:
            max_length = self.config.randomize_positions_from
        else:
            max_length = self.config.block_size
        freqs_cis = precompute_freqs_cis(
            dim // self.config.num_attention_heads,
            max_length,
            self.config.rope_settings.rope_base,  # 50k in the newer models
            self.config.rope_settings.rope_condense_ratio,
        )  # can actually be a buffer now, and remains in fp32! (at least in the settings I tested)
        return freqs_cis

    def reset_parameters(self) -> None:
        self.config.init.apply(self.transformer.wte, "embedding")
        self.config.init.apply(self.transformer.ln_f, "normalization")
        # lm_head init already defined above

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
        return_logits: bool = False,
        num_steps_pair: Optional[torch.Tensor] = None,
    ) -> dict[str, Optional[torch.Tensor]]:
        if self.config.randomize_positions_from is not None and self.training:
            position_ids = torch.sort(  # need to fork rng for distributed
                torch.randint(0, self.config.randomize_positions_from, (input_ids.shape[1],), device=input_ids.device)
            )[0]

        if position_ids is None:
            freqs_cis = self.freqs_cis[:, : input_ids.shape[1]]
        else:
            freqs_cis = self.freqs_cis.index_select(1, position_ids)

        input_embeds = self.transformer.wte(input_ids)
        if self.emb_scale != 1:
            input_embeds = input_embeds * self.emb_scale

        for _, block in enumerate(self.transformer.prelude):
            input_embeds = block(input_embeds, freqs_cis, attention_mask)

        x, num_steps_no_grad, num_steps_with_grad, xk = self.iterate_forward(
            input_embeds,  # type: ignore
            freqs_cis,
            attention_mask,
            num_steps_pair,
        )
        x_rec_output = x

        for _, block in enumerate(self.transformer.coda):
            if self.gradient_checkpointing and "in-coda" in self.config.activation_checkpoint_impl:
                x = self.config.checkpoint(block, x, freqs_cis, attention_mask)
            else:
                x = block(x, freqs_cis, attention_mask)
        if self.gradient_checkpointing and "in-coda" in self.config.activation_checkpoint_impl:
            x = self.config.checkpoint(self.transformer.ln_f, x)
        else:
            x = self.transformer.ln_f(x)

        if self.monitoring:
            self.monitor_module(x, x_rec_output, xk, input_embeds, num_steps_no_grad, num_steps_with_grad)

        if labels is not None:
            logits = None
            if self.config.use_fused_head == "cce":
                from cut_cross_entropy import linear_cross_entropy  # type: ignore[unusal import]

                loss = linear_cross_entropy(
                    x * self.config.init.logit_scale, self.lm_head.weight, labels, filter_eps="auto"
                )
            elif self.config.use_fused_head == "hhe" or self.config.use_fused_head == "full-triton":
                loss = self.lm_head(x * self.config.init.logit_scale, labels)
            else:
                logits = self.lm_head(x).float() * self.config.init.logit_scale
                loss = torch.nn.functional.cross_entropy(logits.view(-1, logits.shape[-1]), labels.view(-1))
            log_ppl = loss.clone().detach()
            if self.config.mcleish_throttle and self.training:
                loss = loss / torch.as_tensor(num_steps_with_grad, device=loss.device)
            if self.config.elbayad_weighing and self.training:
                t = self.config.mean_recurrence
                weights = torch.arange(1, 16 * t, device=loss.device) ** self.config.elbayad_exponent
                weights /= torch.sum(torch.arange(1, t + 1, device=loss.device) ** self.config.elbayad_exponent, dim=0)
                this_weight = weights[num_steps_no_grad + num_steps_with_grad] / weights[t // 2]
                loss = loss * this_weight
        else:
            if self.config.use_fused_head == "cce":
                logits = self.lm_head(x).float() * self.config.init.logit_scale
            elif self.config.use_fused_head == "full-triton":
                logits = (
                    torch.matmul(
                        x, self.lm_head.weight.T if self.config.tie_embeddings else self.lm_head.weight
                    ).float()
                    * self.config.init.logit_scale
                )
            else:
                logits = self.lm_head(x).float() * self.config.init.logit_scale
            loss, log_ppl = torch.as_tensor(0.0), torch.as_tensor(0.0)

        return {
            "loss": loss,
            "logits": logits if return_logits else None,
            "log_ppl": log_ppl,
        }

    # The repeat happens here
    @torch._dynamo.disable(recursive=False)  # type: ignore
    def iterate_forward(self, input_embeds, freqs_cis, mask, num_steps_pair: Optional[torch.Tensor] = None):
        x = self.initialize_state(input_embeds)

        if num_steps_pair is None:
            num_steps_no_grad, num_steps_with_grad = self.randomized_iteration_sampler()  # type: ignore
        elif len(num_steps_pair) > 1:
            num_steps_no_grad, num_steps_with_grad = num_steps_pair
        else:
            num_steps_no_grad, num_steps_with_grad = num_steps_pair, torch.tensor(0)

        if self.config.randomize_embed_step:
            offset = torch.randint(0, self.config.mean_recurrence * 8, (1,), device=input_embeds.device)
        else:
            offset = 0

        with torch.no_grad():
            # ultra annoying in ddp due to
            # https://discuss.pytorch.org/t/does-distributeddataparallel-work-with-torch-no-grad-and-find-unused-parameters-false/122594
            # for now running with find_unused_params=True enabled even though the graph structure is (technically) clear
            # and all parameters are always used
            for step in range(num_steps_no_grad):
                xk = x
                x = self.core_block_forward(xk, input_embeds, freqs_cis, mask, step + offset)

        for step in range(num_steps_with_grad):
            xk = x
            if self.gradient_checkpointing and "per-iteration" in self.config.activation_checkpoint_impl:
                x = self.config.checkpoint(
                    self.core_block_forward, xk, input_embeds, freqs_cis, mask, num_steps_no_grad + step + offset
                )
            else:
                x = self.core_block_forward(xk, input_embeds, freqs_cis, mask, num_steps_no_grad + step + offset)
        return self.transformer.ln_f(x), num_steps_no_grad, num_steps_with_grad, xk.detach()

    def core_block_forward(self, x, input_embeds, freqs_cis, mask, step: Union[torch.Tensor, int]):
        if self.config.embed_step:
            context = self.step_embedding(torch.as_tensor([step], device=input_embeds.device))
        else:
            context = None

        if self.config.injection_type == "add":
            x = x + input_embeds
        elif self.config.injection_type == "gate":
            x = x * input_embeds
        elif self.config.injection_type in ["linear", "ffn"]:
            x = self.transformer.adapter(torch.cat([x, input_embeds], dim=-1))
        elif self.config.injection_type == "modulated":  # use in conjunction with Modulated blocks, not with embed_step
            context = x.clone()
        else:
            raise ValueError("Invalid injection type")

        if self.config.intermediate_noise_injection > 0:
            n = self.config.intermediate_noise_injection
            if self.config.geom_noise_injection == "geom":
                step1 = torch.as_tensor(step + 1, device=x.device)  # need to cast for compile
                x = x * (1 - n / step1) + torch.randn_like(x) * n / step1
            elif self.config.geom_noise_injection == "sqrt":
                step1sqrt = torch.as_tensor(step + 1, device=x.device).sqrt()  # need to cast for compile
                x = x * (1 - n / step1sqrt) + torch.randn_like(x) * n / step1sqrt
            elif self.config.geom_noise_injection == "line":
                noise = max(n, (self.config.maximal_recurrence - step) / self.config.maximal_recurrence)  # type: ignore
                x = x * (1 - noise) + torch.randn_like(x) * noise
            elif self.config.geom_noise_injection == "chi":
                noise = 2 * torch.rand(1, device=x.device, dtype=x.dtype) * n
            else:
                x = x * (1 - n) + torch.randn_like(x) * n

        if isinstance(self.transformer.core_block[0], ModulatedTransformerPostNormBlock):
            for _, block in enumerate(self.transformer.core_block):
                if not self.gradient_checkpointing:
                    x = block(x, freqs_cis, mask, context=context)
                else:
                    x = self.config.checkpoint(block, x, freqs_cis, mask, context=context)
        else:
            if context is not None:
                x = x + context

            for _, block in enumerate(self.transformer.core_block):
                if self.gradient_checkpointing and "per-block" in self.config.activation_checkpoint_impl:
                    x = self.config.checkpoint(block, x, freqs_cis, mask)
                else:
                    x = block(x, freqs_cis, mask)
        return x

    @torch._dynamo.disable(recursive=False)  # type: ignore
    def randomized_iteration_sampler(self) -> tuple[torch.Tensor, torch.Tensor]:
        """Outputs are long tensors so that they can be passed through compiled functions"""
        if torch.rand((1,)).is_meta:  # annoying clause to make meta-tensor-based flop counting work
            # these values are only approximate, not all schemes exactly target a mean of n and k
            # they overvalue the compute done when curricula are turned on, but that may be considered
            # a feature, given that it is a valid form of training acceleration
            return self.config.mean_recurrence - self.config.mean_backprop_depth, self.config.mean_backprop_depth  # type: ignore

        seed_n = 514229 + self.step  # easiest way to make the sampler re-runnable in checkpointing
        seed_k = 317811 + self.step
        if not self.config.lockstep_n and torch.distributed.is_initialized():
            seed_n = seed_n * (torch.distributed.get_rank() + 1)
        if not self.config.lockstep_k and torch.distributed.is_initialized():
            seed_k = seed_k * (torch.distributed.get_rank() + 1)

        n_generator = torch.Generator(device="cpu")
        n_generator.manual_seed(seed_n % (2**31 - 1))
        k_generator = torch.Generator(device="cpu")
        k_generator.manual_seed(seed_k % (2**31 - 1))

        if "curriculum-" in self.config.sampling_scheme:
            ramp_length = int(self.config.sampling_scheme.split("curriculum-")[1])
            if self.step > ramp_length:
                t = max(self.config.mean_recurrence - self.config.mean_backprop_depth, 0)
                s = self.config.mean_backprop_depth
            else:
                slope = self.step / ramp_length
                t = max(math.ceil(slope * (self.config.mean_recurrence - self.config.mean_backprop_depth)), 0)
                s = max(math.ceil(slope * self.config.mean_backprop_depth), 1)
        else:
            t = max(self.config.mean_recurrence - self.config.mean_backprop_depth, 0)
            s = self.config.mean_backprop_depth

        if self.training:
            if "bptt" in self.config.sampling_scheme:  # skewed toward n+k ~ max_recurrence
                n = torch.randint(low=0, high=t * 2, size=(1,), generator=n_generator)
                k = torch.randint(low=1, high=1 + min(t * 2 - int(n.item()), s * 2), size=(1,), generator=k_generator)
            elif "non-uniform" in self.config.sampling_scheme:  # n+k ~ uniform, n ~ 1
                n_plus_k = torch.randint(low=0, high=2 * t, size=(1,), generator=n_generator)
                k = torch.randint(low=1, high=2 * min(t, s) + 1, size=(1,), generator=k_generator)
                n = torch.clamp(n_plus_k - k, min=0)
            elif "gupta" in self.config.sampling_scheme:  # skewed toward n+k ~ uniform, k ~ 1
                # https://github.com/aks2203/deep-thinking/issues/10
                n = torch.randint(low=0, high=t * 2, size=(1,), generator=n_generator)
                draw = torch.rand(size=(1,), generator=k_generator)
                skew = torch.randint(low=2 * t, high=t * 8, size=(1,), generator=k_generator)
                k = 1 + (t - n) * draw**skew
            elif "simple" in self.config.sampling_scheme:  # me not make complicate? n + k ~trapezoidal
                n = torch.randint(low=0, high=2 * t, size=(1,), generator=n_generator)
                k = torch.randint(low=1, high=2 * s + 1, size=(1,), generator=k_generator)
            elif "poisson-lognormal-filling" in self.config.sampling_scheme:
                sigma = 0.5
                mu = math.log(t + s) - (sigma**2 / 2)
                rate = torch.zeros((1,)).log_normal_(mean=mu, std=sigma, generator=n_generator)
                p = torch.poisson(torch.tensor([rate], dtype=torch.float), generator=n_generator) + 1
                n = torch.clamp(p - s, min=0)
                k = torch.as_tensor(torch.minimum(torch.as_tensor(s), p))
            elif "poisson-lognormal-fill" in self.config.sampling_scheme:
                sigma = 0.5
                mu = math.log(t) - (sigma**2 / 2)
                rate = torch.zeros((1,)).log_normal_(mean=mu, std=sigma, generator=n_generator)
                n = torch.poisson(torch.tensor([rate], dtype=torch.float), generator=n_generator)
                k = torch.as_tensor(s)
            elif "poisson-lognormal" in self.config.sampling_scheme:
                sigma = 0.5
                mu = math.log(t) - (sigma**2 / 2)
                rate = torch.zeros((1,)).log_normal_(mean=mu, std=sigma, generator=n_generator)
                n = torch.poisson(torch.tensor([rate], dtype=torch.float), generator=n_generator)
                k = torch.randint(1, 2 * s + 1, (1,), generator=k_generator)
            elif "poisson-unbounded" in self.config.sampling_scheme:
                n = torch.poisson(torch.tensor([t], dtype=torch.float), generator=n_generator)
                k = torch.randint(low=1, high=2 * s + 1, size=(1,), generator=k_generator)
            elif "poisson-fill" in self.config.sampling_scheme:
                n = torch.poisson(torch.tensor([t], dtype=torch.float), generator=n_generator)
                k = torch.as_tensor(s)
            elif "poisson-bounded" in self.config.sampling_scheme:
                n = torch.minimum(
                    torch.poisson(torch.tensor([t], dtype=torch.float), generator=n_generator),
                    torch.as_tensor(2 * t - 1),
                )
                k = torch.randint(low=1, high=2 * s + 1, size=(1,), generator=k_generator)
            elif "negative-binomial" in self.config.sampling_scheme:
                n = torch.as_tensor(sample_negative_binomial(2 * t, t))
                k = torch.randint(1, 2 * s + 1, (1,))
            elif "sobol" in self.config.sampling_scheme:  # this is sobol+simple
                nk_generator = torch.quasirandom.SobolEngine(dimension=2, scramble=True, seed=seed_n)
                n_, k_ = nk_generator.draw(1).flatten()
                n = (n_ * 2 * t).to(torch.long)
                k = (k_ * 2 * s + 1).to(torch.long)
            elif "geometric" in self.config.sampling_scheme:
                n = torch.as_tensor(1.0).geometric_(1 / t, generator=n_generator)
                k = torch.randint(low=1, high=2 * s + 1, size=(1,), generator=k_generator)
            elif "fixed" in self.config.sampling_scheme:
                n, k = torch.as_tensor(t), torch.as_tensor(s)
            elif "non-recurrent" in self.config.sampling_scheme:
                n, k = torch.as_tensor(0), torch.as_tensor(1)
            elif "full" in self.config.sampling_scheme:
                n, k = torch.as_tensor(0), torch.randint(low=1, high=2 * s + 1, size=(1,), generator=k_generator)
        else:
            n, k = torch.as_tensor(self.config.mean_recurrence), torch.as_tensor(0)

        return n.to(dtype=torch.long), k.to(dtype=torch.long)

    def initialize_state(self, input_embeds):
        if self.config.injection_type == "none":
            return input_embeds
        if self.config.state_init == "normal":
            x = torch.randn_like(input_embeds)
        elif self.config.state_init == "embed":  # initialized like a scaled embedding:
            x = torch.randn_like(input_embeds).mul(1 / math.sqrt(input_embeds.shape[-1]))
        elif self.config.state_init == "like-init":
            x = torch.randn_like(input_embeds)
            std = self.config.init.get_std("embedding")
            torch.nn.init.trunc_normal_(x, mean=0.0, std=std, a=-3 * std, b=3 * std)
            if self.emb_scale != 1:
                x = x * self.emb_scale
        elif self.config.state_init == "zero":
            x = torch.zeros_like(input_embeds)
        elif self.config.state_init == "unit":
            x = torch.randn_like(input_embeds)
            std, mean = torch.std_mean(x, dim=-1, keepdim=True)
            x = (x - mean) / std
        return x

    @torch.no_grad()
    def monitor_module(
        self,
        x_out: torch.Tensor,
        x_rec: torch.Tensor,
        xk: torch.Tensor,
        input_embeds: torch.Tensor,
        num_steps_no_grad: torch.Tensor,
        num_steps_with_grad: torch.Tensor,
    ):
        """Should update to track more recurrence metrics"""
        x_out_c = x_out - x_out.mean(dim=-1, keepdim=True)
        normed_x = x_out_c / x_out_c.norm(dim=-1, keepdim=True)
        token_corr = (normed_x @ normed_x.transpose(1, 2)).mean() - 1 / x_out.shape[1]

        x_rec_c = x_rec - x_rec.mean(dim=-1, keepdim=True)
        normed_x = x_rec_c / x_rec_c.norm(dim=-1, keepdim=True)
        token_corr_rec = (normed_x @ normed_x.transpose(1, 2)).mean() - 1 / x_rec.shape[1]
        # k = num_steps_no_grad + num_steps_with_grad
        metrics = {
            "last_hidden_token_corr": token_corr,
            "recurrent_state_token_corr": token_corr_rec,
            "last_hidden_norm": x_out.norm(dim=-1).mean(),
            "recurrent_state_norm": x_rec.norm(dim=-1).mean(),
            "recurrent_diff": (x_rec - input_embeds).norm(dim=-1).mean(),
            "num_steps_no_grad": num_steps_no_grad,
            "num_steps_with_grad": num_steps_with_grad,
            "recurrent_residual": (x_rec - xk).norm(dim=-1).mean(),
            "rel_residual": ((x_rec - xk).norm(dim=-1) / x_rec.norm(dim=-1)).mean(),
            # f"rel_residual_at_{k}": ((x_rec - xk).norm(dim=-1) / x_rec.norm(dim=-1)).mean(),
        }
        self.latest_metrics = metrics  # will be picked up from monitoring caller

# Simplified Version of the iteration

In [ ]:

def iterate_forward(self, input_embeds, freqs_cis, mask, num_steps_pair: Optional[torch.Tensor] = None):
    """
    Unroll the core_block multiple times. Some steps may be done with no_grad,
    then a few steps with gradients, for training efficiency.
    """
    # Initialize the latent state
    x = self.initialize_state(input_embeds)

    # Decide how many steps have no grad vs. gradient
    if num_steps_pair is None:
        num_steps_no_grad, num_steps_with_grad = self.randomized_iteration_sampler()
    elif len(num_steps_pair) > 1:
        num_steps_no_grad, num_steps_with_grad = num_steps_pair
    else:
        num_steps_no_grad, num_steps_with_grad = num_steps_pair, torch.tensor(0)

    # (Optional) offset if config.randomize_embed_step
    if self.config.randomize_embed_step:
        offset = torch.randint(0, self.config.mean_recurrence * 8, (1,), device=input_embeds.device)
    else:
        offset = 0

    # ---------------------- NO-GRAD STEPS ----------------------
    with torch.no_grad():
        for step in range(num_steps_no_grad):
            xk = x
            x = self.core_block_forward(xk, input_embeds, freqs_cis, mask, step + offset)

    # ---------------------- WITH-GRAD STEPS ----------------------
    for step in range(num_steps_with_grad):
        xk = x
        if (self.gradient_checkpointing and
            "per-iteration" in self.config.activation_checkpoint_impl):
            x = self.config.checkpoint(
                self.core_block_forward,
                xk, input_embeds, freqs_cis, mask, num_steps_no_grad + step + offset
            )
        else:
            x = self.core_block_forward(xk, input_embeds, freqs_cis, mask, num_steps_no_grad + step + offset)

    # Apply final normalization to the recurrent output
    x = self.transformer.ln_f(x)
    return x, num_steps_no_grad, num_steps_with_grad, xk.detach()

def core_block_forward(self, x, input_embeds, freqs_cis, mask, step: Union[torch.Tensor, int]):
    """
    One 'iteration' of the core block. Possibly merges current state x
    with the input embedding in various ways ('add', 'gate', or 'ffn').
    Then runs the 'core_block' layers in sequence.
    """

    # Optionally embed the current iteration step as 'context'
    if self.config.embed_step:
        context = self.step_embedding(torch.as_tensor([step], device=input_embeds.device))
    else:
        context = None

    # ---------------------- MERGE (INJECTION) ----------------------
    if self.config.injection_type == "add":
        x = x + input_embeds
    elif self.config.injection_type == "gate":
        x = x * input_embeds
    elif self.config.injection_type in ["linear", "ffn"]:
        # If "ffn", self.transformer.adapter is actually an MLP that processes concat([x, input_embeds])
        x = self.transformer.adapter(torch.cat([x, input_embeds], dim=-1))
    elif self.config.injection_type == "modulated":
        context = x.clone()
    else:
        raise ValueError("Invalid injection type")

    # (Optional) Add noise if configured...
    if self.config.intermediate_noise_injection > 0:
        # ... apply whichever schedule is indicated ...
        pass

    # ---------------------- APPLY 'core_block' LAYERS ----------------------
    if isinstance(self.transformer.core_block[0], ModulatedTransformerPostNormBlock):
        # If the blocks are "modulated," we pass 'context' into them
        for _, block in enumerate(self.transformer.core_block):
            if not self.gradient_checkpointing:
                x = block(x, freqs_cis, mask, context=context)
            else:
                x = self.config.checkpoint(block, x, freqs_cis, mask, context=context)
    else:
        # Normal path: add 'context' if we are using a step embedding, then call each layer
        if context is not None:
            x = x + context
        for _, block in enumerate(self.transformer.core_block):
            if self.gradient_checkpointing and "per-block" in self.config.activation_checkpoint_impl:
                x = self.config.checkpoint(block, x, freqs_cis, mask)
            else:
                x = block(x, freqs_cis, mask)

    return x